# SFINCS-JAX Neoclassical Cached Demo

## What you will learn
What a richer drift-kinetic validation calculation adds beyond a compact effective-ripple metric.

## Codes used
`sfincs_jax` in research mode; cached D11/Er/bootstrap curves by default.

## Run mode
This notebook uses RUN_MODE = "cached" by default. Allowed values are "tiny", "cached", and "research".

## Expected outputs
`07_sfincs_d11_scan.png`, `07_er_roots.png`, and `07_bootstrap_profile.png`.

In [ ]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = None
for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "src" / "sos2026").exists():
        PROJECT_ROOT = candidate
        break
if PROJECT_ROOT is None:
    PROJECT_ROOT = Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))
print(f"Project root: {PROJECT_ROOT}")

In [ ]:
try:
    import google.colab  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB:
    print("Colab detected. Keep RUN_MODE='cached' first; install requirements-colab.txt from the cloned repo if needed.")
else:
    print("Local runtime detected.")

In [ ]:
RUN_MODE = "cached"  # allowed: "tiny", "cached", "research"
print(f"RUN_MODE = {RUN_MODE}")

In [ ]:
import importlib
import json
import math
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sos2026.paths import PROJECT_ROOT, DATA_DIR, CACHE_DIR, FIGURE_DIR, MOVIE_DIR, ensure_directories
ensure_directories()
print("Figures:", FIGURE_DIR)
print("Cached data:", CACHE_DIR)

## 1. Learning frame

This notebook is a deliberately small project: define one metric, produce one plot, expose one failure mode, and identify where a real code would enter.

In [ ]:
from sos2026.transport_helpers import sfincs_like_curves
from sos2026.plotting import savefig, caption

## 2. Load or generate the teaching data

Cached mode uses small arrays so the conceptual workflow is always available.

In [ ]:
d = sfincs_like_curves()
pd.DataFrame({"nu": d["nu"], "D11_no_Er": d["D11_no_Er"], "D11_with_Er": d["D11_with_Er"]}).head()

## 3. Make the primary plot

Every plot has a one-sentence caption because students should know how to read it without guessing.

In [ ]:
fig, ax = plt.subplots(figsize=(6.2, 3.8))
ax.loglog(d["nu"], d["D11_no_Er"], lw=2, label="without Er suppression")
ax.loglog(d["nu"], d["D11_with_Er"], lw=2, label="with Er suppression")
ax.set_xlabel("collisionality proxy")
ax.set_ylabel("D11-like coefficient")
ax.set_title("Cached neoclassical scan")
ax.legend(fontsize=8)
ax.grid(True, which="both", alpha=0.25)
caption(ax, "This teaches the shape of a validation scan; it is not a production SFINCS_JAX solve.")
savefig(fig, "07_sfincs_d11_scan.png")
plt.show()

## 4. Probe the metric

A metric becomes useful for optimization only when we understand how it changes across design choices.

In [ ]:
fig, ax = plt.subplots(figsize=(6.0, 3.6))
ax.plot(d["Er"], d["ambipolar"], color="#b42318", lw=2)
ax.axhline(0, color="black", lw=0.8)
ax.set_xlabel("radial electric field proxy")
ax.set_ylabel("ion flux - electron flux")
ax.set_title("Ambipolar Er roots")
ax.grid(alpha=0.25)
caption(ax, "Roots mark candidate ambipolar electric fields that affect impurities and turbulence.")
savefig(fig, "07_er_roots.png")
plt.show()

## 5. Interpret the design consequence

The table below translates the plot into an optimization decision.

In [ ]:
fig, ax = plt.subplots(figsize=(6.0, 3.6))
ax.plot(d["s"], d["bootstrap"], lw=2, color="#0f766e")
ax.set_xlabel("s")
ax.set_ylabel("bootstrap-current proxy")
ax.set_title("Bootstrap-current constraint")
ax.grid(alpha=0.25)
caption(ax, "Bootstrap current is a profile-dependent constraint, not just a geometry score.")
savefig(fig, "07_bootstrap_profile.png")
plt.show()

## 6. Failure mode

The cached plot is useful only if we say what it does not prove.

In [ ]:
failure_mode = pd.DataFrame({
    "cached_mode_proves": ["workflow shape", "plot grammar", "where the metric enters"],
    "cached_mode_does_not_prove": ["validated physics", "final design ranking", "runtime scalability"],
})
failure_mode

## 7. Research-mode hook

Run this cell only after timing the package on the lecture machine.

In [ ]:
if RUN_MODE == "research":
    import sfincs_jax
    print("sfincs_jax import OK:", getattr(sfincs_jax, "__version__", "unknown"))
else:
    print("Cached mode: research package path skipped intentionally.")

## 8. Mini project handoff

Use this notebook during the lecture as the computational project slide points to: change one parameter, regenerate one plot, and explain one design tradeoff.

In [ ]:
project_steps = pd.DataFrame({
    "step": [1, 2, 3, 4],
    "action": ["identify metric", "change one input", "regenerate plot", "state failure mode"],
})
project_steps

## Try this
Change one scalar or one row in the cached data and regenerate the primary plot.

## Expected qualitative answer
The plot should move in a physically interpretable direction, but the cached result remains an educational proxy.

## Research extension
Replace the cached data source with the corresponding real package output after timing and API verification.